# 17 — Output Guardrails and Moderation

PII detection/redaction, content moderation, and output validation with retry.

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'

In [ ]:
import re
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

## Guardrail 1: PII Detection

In [ ]:
PII_PATTERNS = {
    "email": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b",
    "phone": r"\b(?:\+?1[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b",
    "ssn": r"\b\d{3}-\d{2}-\d{4}\b",
    "credit_card": r"\b\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b",
}

def detect_pii(text):
    findings = []
    for pii_type, pattern in PII_PATTERNS.items():
        for match in re.findall(pattern, text):
            findings.append({"type": pii_type, "value": match})
    return findings

def redact_pii(text):
    redacted = text
    for pii_type, pattern in PII_PATTERNS.items():
        redacted = re.sub(pattern, f"[REDACTED_{pii_type.upper()}]", redacted)
    return redacted

for text in [
    "Contact John at john.doe@example.com or call 555-123-4567.",
    "My SSN is 123-45-6789 and my card is 4111 1111 1111 1111.",
    "The meeting is at 3pm in conference room B.",
]:
    pii = detect_pii(text)
    print(f"[{len(pii)} PII items] {text}")
    if pii:
        print(f"  Redacted: {redact_pii(text)}")

## Guardrail 2: Content Moderation

In [ ]:
class ModerationResult(BaseModel):
    is_safe: bool
    category: str
    reason: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
mod_chain = ChatPromptTemplate.from_template(
    "Assess whether this text is safe for a general audience.\n\nText: {text}"
) | llm.with_structured_output(ModerationResult)

for content in ["Python is a great programming language.", "Here are tips for improving code quality."]:
    result = mod_chain.invoke({"text": content})
    print(f"[{'SAFE' if result.is_safe else 'FLAGGED'}] {content[:60]}... — {result.reason}")

## Guardrail 3: Output Validation with Retry

In [ ]:
class ValidatedAnswer(BaseModel):
    answer: str
    confidence: float
    sources_cited: bool

val_chain = ChatPromptTemplate.from_template(
    "Answer accurately and cite your reasoning.\n\nQuestion: {question}"
) | llm.with_structured_output(ValidatedAnswer)

for q in ["What is the speed of light in a vacuum?", "When was Python first released?"]:
    for attempt in range(3):
        result = val_chain.invoke({"question": q})
        if result.confidence >= 0.7:
            print(f"Q: {q}\nA: [{result.confidence:.0%}] {result.answer}\n")
            break
        print(f"  Retry {attempt+1}: low confidence ({result.confidence:.0%})")